# Soninké TTS Libre — essai OmniVoice reproductible (V2)

Objectif : tenter une vraie génération WAV pour trois phrases de référence, sans modifier le dépôt et sans entraînement. Le résultat n'est validé qu'après écoute humaine.

**Coût et licences :** n'active aucun forfait payant. Si Colab n'accorde pas de GPU gratuit, arrête et réessaie plus tard. Les poids préentraînés OmniVoice sont indiqués sous licence non commerciale ; ce test ne vaut pas autorisation d'usage commercial. Le code et les poids ont des licences distinctes.

**Confidentialité :** ce notebook n'importe pas automatiquement le corpus privé et ne publie aucun audio sur GitHub. Les fichiers générés sont enregistrés dans le dossier de sortie Google Drive choisi ci-dessous.

## 1. Installation

Exécute cette cellule une fois dans une nouvelle session Colab. L'installation peut prendre du temps.

In [ ]:
%pip -q install omnivoice soundfile numpy


## 2. Vérifier le matériel et préparer le dossier

Dans Colab, sélectionne **Environnement d'exécution → Modifier le type d'exécution → GPU** uniquement si l'option gratuite est disponible. Aucun calcul CPU n'est lancé si le GPU manque.

In [ ]:
import os, json, datetime, zipfile, hashlib
from pathlib import Path
import numpy as np
import torch
import soundfile as sf
from IPython.display import Audio, display

if not torch.cuda.is_available():
    raise RuntimeError('ARRÊT SÛR : aucun GPU disponible dans cette session. Ne paie pas pour continuer ; réessaie plus tard avec une session gratuite.')
print('GPU :', torch.cuda.get_device_name(0))
print('PyTorch :', torch.__version__)

OUT_DIR = Path('/content/soninke_tts_test_v2')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dossier de sortie temporaire :', OUT_DIR)


## 3. Charger le modèle et vérifier la langue

Le chargement peut télécharger plusieurs gigaoctets. Cette cellule affiche si `soninke`/`snk` apparaît dans les langues reconnues. Une présence dans la liste ne garantit pas la qualité de la prononciation.

In [ ]:
from omnivoice import OmniVoice
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda:0', dtype=torch.float16)
lang_names = list(model.supported_language_names())
lang_ids = list(model.supported_language_ids())
soninke_name_matches = [x for x in lang_names if 'sonink' in str(x).lower()]
snk_id_matches = [x for x in lang_ids if str(x).lower() == 'snk']
print('Fréquence audio (Hz):', model.sampling_rate)
print('Noms correspondant à Soninke:', soninke_name_matches)
print('Code snk présent:', bool(snk_id_matches))
if not soninke_name_matches and not snk_id_matches:
    raise RuntimeError('ARRÊT : cette version chargée ne confirme pas Soninke/snk. Ne retire pas le paramètre de langue ; relève la version et l’erreur.')


## 4. Générer les phrases de référence

Les chaînes ci-dessous sont conservées séparément, sans traduction ni correction automatique. Chaque WAV nouvellement produit reste en attente de validation humaine, même si une phrase proche a déjà été validée auparavant.

In [ ]:
phrases = [
    ('daga_yige', 'Daga yige'),
    ('li_taxu', 'Li taxu'),
    ('daga_goli', 'Daga goli'),
]
manifest = []
for sample_id, phrase in phrases:
    print(f'Génération : {sample_id} — {phrase}')
    try:
        generated = model.generate(text=phrase, language='soninke')
        if not generated or np.asarray(generated[0]).size == 0:
            raise RuntimeError('Le moteur a renvoyé un audio vide.')
        samples = np.asarray(generated[0]).squeeze()
        if samples.ndim != 1 or not np.isfinite(samples).all():
            raise RuntimeError('Audio invalide : forme inattendue ou valeurs non finies.')
        wav_path = OUT_DIR / f'{sample_id}.wav'
        sf.write(str(wav_path), samples, int(model.sampling_rate), subtype='PCM_16')
        info = sf.info(str(wav_path))
        entry = {
            'id': sample_id, 'text_exact': phrase, 'language_argument': 'soninke',
            'model_id': 'k2-fsa/OmniVoice', 'sample_rate_hz': int(info.samplerate),
            'channels': int(info.channels), 'duration_seconds': round(info.duration, 3),
            'file': wav_path.name, 'sha256': hashlib.sha256(wav_path.read_bytes()).hexdigest(),
            'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'human_validation': 'pending'
        }
        manifest.append(entry)
        print('WAV créé :', wav_path, '| durée:', info.duration, 's')
        display(Audio(str(wav_path)))
    except Exception as exc:
        print(f'ÉCHEC pour {sample_id}: {type(exc).__name__}: {exc}')
        manifest.append({'id': sample_id, 'text_exact': phrase, 'human_validation': 'not_generated', 'error': f'{type(exc).__name__}: {exc}'})
        break

manifest_path = OUT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Manifeste :', manifest_path)
print('Générations avec WAV :', sum(1 for x in manifest if x.get('file')),'/',len(phrases))


## 5. Sauvegarder dans Google Drive et créer une archive

Cette étape n'est exécutée que si tu la lances. Elle copie les résultats générés vers un dossier de ton Drive personnel ; elle ne publie rien sur le dépôt GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = Path('/content/drive/MyDrive/SONINKE_TTS_LIBRE/TEST_OMNIVOICE_V2')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
for item in OUT_DIR.iterdir():
    if item.is_file():
        (DRIVE_OUT / item.name).write_bytes(item.read_bytes())
zip_path = DRIVE_OUT / 'soninke_tts_test_v2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in DRIVE_OUT.iterdir():
        if item.is_file() and item != zip_path:
            zf.write(item, arcname=item.name)
print('Résultats sauvegardés dans :', DRIVE_OUT)
print('Archive :', zip_path)


## 6. Grille de validation humaine

Pour chaque fichier, le référent linguistique doit écouter et noter séparément :
- **Clarté audio** : clair / à corriger.
- **Prononciation soninké** : correcte / incorrecte / incertaine.
- **Correspondance avec le texte** : exacte / écart constaté.
- **Naturel et prosodie** : acceptable / à corriger.

Ne marque aucun fichier comme validé sans l'avoir écouté. Conserve les sorties ratées et leurs messages d'erreur dans le compte rendu. Une génération réussie techniquement n'est pas une preuve de qualité linguistique.